# 01. YouTube Data Collection & Ingestion Engine
## Production Data Science Pipeline
This notebook demonstrates:
1. YouTube Data API v3 and YouTube Analytics API v2 client configuration.
2. Quota allocation and rate limiting governance (10,000 units/day budget).
3. Compliance verification with YouTube Developer Policies (Section III.E.4 30-day raw retention TTL).
4. Extracting channel summary, video metadata, and transactional warehouse loading.


In [ ]:
import os
import sys
import sqlite3
import pandas as pd
from datetime import datetime, timezone, timedelta

# Project Root Setup
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

DB_PATH = os.path.join(PROJECT_ROOT, "youtube_growth.db")
print(f"Connecting to Database: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
df_channels = pd.read_sql_query("SELECT * FROM channels", conn)
print(f"Loaded {len(df_channels)} channel records.")
df_channels.head()


In [ ]:
# Inspect Raw Video Records with Mandatory Compliance Expiry Timestamps
df_videos = pd.read_sql_query("""
    SELECT video_id, title, duration_seconds, published_at, retrieved_at, youtube_data_expires_at, retention_policy
    FROM videos
    LIMIT 5
""", conn)
df_videos


### Policy Compliance Verification (Section III.E.4)
Every raw record has a `youtube_data_expires_at` timestamp set to exactly `retrieved_at + 30 days`.
The platform's compliance daemon ensures that all raw telemetry past 30 days is automatically purged or aggregated into derived statistical features.
